In [1]:
import pandas as pd
import re
import os
DB_OUTPUT = f"dataset\database_v3"
os.makedirs(DB_OUTPUT, exist_ok=True)

In [2]:
df = pd.read_csv(r"dataset\URA_merged_full_v3_removed_outliers.csv")
print(df['Project Name'].nunique())
print(len(df.columns.tolist()))
for col in df.columns:
    print(col)

C:\Users\cecel\AppData\Local\Temp\ipykernel_36728\3580349771.py:1: DtypeWarning: Columns (0: Completion Date, 1: Tenure Start Date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"dataset\URA_merged_full_v3_removed_outliers.csv")


2497
340
Project Name
Street Name
Postal District
Property Type
No of Bedroom
Monthly Rent ($)
Floor Area (SQM)
Floor Area (SQFT)
Lease Commencement Date
district
Completion Date
Postal Sector
Planning Region
Planning Area
Tenure Length
Tenure Start Date
year
month
year_month
onemap_address
latitude
longitude
Nanyang Primary School
Rosyth School
Henry Park Primary School
Tao Nan School
Raffles Girls' Primary School
St. Hilda's Primary School
Pei Hwa Presbyterian Primary School
Methodist Girls' School (Primary)
Nan Hua Primary School
Chij St. Nicholas Girls' School
Anglo-Chinese School (Primary)
Catholic High School (Primary)
Rulanh Primary School
Red Swastika School
Ai Tong School
St. Joseph's Institution Junior
Kong Hwa School
South View Primary School
Chongfu School
Pei Chun Public School
Holy Innocents' Primary School
Maris Stella High School (Primary)
Singapore Chinese Girls' Primary School
Canberra Primary School
Radin Mas Primary School
River Valley Primary School
Gongshang Prima

In [3]:
df.head()

,Project Name,Street Name,Postal District,Property Type,No of Bedroom,Monthly Rent ($),Floor Area (SQM),Floor Area (SQFT),Lease Commencement Date,district,...,2nd Post Offices dist,3rd Post Offices,3rd Post Offices dist,4th Post Offices,4th Post Offices dist,5th Post Offices,5th Post Offices dist,Project Name in Realis,Floor Area (SQFT) mid,Rent per SQFT
0,ALLAN VILLE,GEMMILL LANE,1,Non-landed Properties,NaN,4500,100 to 110,"1,000 to 1,100",2012-02-01,D01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1050.0,4.285714
1,ALLAN VILLE,GEMMILL LANE,1,Non-landed Properties,NaN,2750,50 to 60,600 to 700,2012-07-01,D01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,650.0,4.230769
2,ALLAN VILLE,GEMMILL LANE,1,Non-landed Properties,NaN,2200,30 to 40,400 to 500,2012-08-01,D01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450.0,4.888889
3,ALLAN VILLE,GEMMILL LANE,1,Non-landed Properties,NaN,2850,50 to 60,600 to 700,2012-08-01,D01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,650.0,4.384615
4,ALLAN VILLE,GEMMILL LANE,1,Non-landed Properties,NaN,2800,60 to 70,600 to 700,2012-08-01,D01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,650.0,4.307692


In [4]:
df['Lease Commencement Date'] = pd.to_datetime(df['Lease Commencement Date'], errors='coerce')
df = df.dropna(subset=['Lease Commencement Date'])
df = df.sort_values('Lease Commencement Date')
df['timestep'] = df['Lease Commencement Date'].rank(method='dense').astype(int) - 1
df['transaction_id'] = [i for i in range(1, len(df) + 1)]
df['project_id'] = df['Project Name'].astype('category').cat.codes

C:\Users\cecel\AppData\Local\Temp\ipykernel_36728\3658964363.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['timestep'] = df['Lease Commencement Date'].rank(method='dense').astype(int) - 1
C:\Users\cecel\AppData\Local\Temp\ipykernel_36728\3658964363.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['transaction_id'] = [i for i in range(1, len(df) + 1)]
C:\Users\cecel\AppData\Local\Temp\ipykernel_36728\3658964363.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

In [5]:
project_df = (
    df[['project_id', 'Project Name']]
    .drop_duplicates()
    .sort_values('project_id')
)

project_df.to_csv(os.path.join(DB_OUTPUT, "project_id.csv"), index=False)

In [6]:
timestep_df = (
    df[['timestep', 'Lease Commencement Date']]
    .drop_duplicates()
    .sort_values('timestep')
)

timestep_df.to_csv(os.path.join(DB_OUTPUT, "timestep.csv"), index=False)

In [7]:
# df['sqft'] = (
#     df['Floor Area (SQFT)']
#     .astype(str)
#     .str.replace(',', '', regex=False)          # remove commas
#     .str.extract(r'(\d+)\s*to\s*(\d+)')         # extract numbers
#     .astype(float)
#     .mean(axis=1)
# )
# df['rent_per_sqft'] = df['Monthly Rent ($)'] / df['sqft']

## V3
df = df.rename(columns={
    "Floor Area (SQFT) mid": "sqft",
    "Rent per SQFT": "rent_per_sqft"
})

In [8]:
transaction_df = (
    df[['transaction_id', 'timestep', 'Lease Commencement Date', 'project_id', 'Project Name', 'No of Bedroom', 'Floor Area (SQFT)', 'sqft', 'Monthly Rent ($)', 'rent_per_sqft']]
    .drop_duplicates()
    .sort_values('transaction_id')
)

transaction_df.to_csv(os.path.join(DB_OUTPUT, "transaction.csv"), index=False)

In [9]:
macro_cols = [
    'timestep',
    'Lease Commencement Date',
    'cpi_all_items_infl_yoy_pct',
    'unemployment_rate_sa_pct',
    'sora_overnight_pct',
    'sora_compounded_3m_pct',
    'sg_govt_bond_yield_10y_pct',
    'sgd_per_usd_logret_12m_pct',
    'ura_private_rental_index_yoy_pct',
    'ura_private_price_index_avg_3seg_yoy_pct',
    'hdb_resale_price_index_yoy_pct',
    'gva_yoy_growth_pct',
    'cpi_all_items_index',
    'unemployment_rate_sa_pct_monthly',
    'sora_overnight_pct_monthly',
    'sora_compounded_3m_pct_monthly',
    'sg_govt_bond_yield_10y_pct_monthly',
    'sgd_per_usd',
    'ura_private_rental_index',
    'ura_private_price_index_ccr',
    'ura_private_price_index_rcr',
    'ura_private_price_index_ocr',
    'hdb_resale_price_index',
    'gva_yoy_growth_pct_monthly',
    'cpi_all_items_infl_mom_pct',
    'cpi_all_items_infl_yoy_pct_monthly',
    'ura_private_rental_index_qoq_pct',
    'ura_private_rental_index_yoy_pct_monthly',
    'ura_private_price_index_ccr_qoq_pct',
    'ura_private_price_index_ccr_yoy_pct',
    'ura_private_price_index_rcr_qoq_pct',
    'ura_private_price_index_rcr_yoy_pct',
    'ura_private_price_index_ocr_qoq_pct',
    'ura_private_price_index_ocr_yoy_pct',
    'hdb_resale_price_index_qoq_pct',
    'hdb_resale_price_index_yoy_pct_monthly',
    'ura_private_price_index_avg_3seg',
    'ura_private_price_index_avg_3seg_qoq_pct',
    'ura_private_price_index_avg_3seg_yoy_pct_monthly',
    'sgd_per_usd_logret_1m_pct',
    'sgd_per_usd_logret_12m_pct_monthly',
    'sora_overnight_pct_chg_bp',
    'sora_compounded_3m_pct_chg_bp',
    'sg_govt_bond_yield_10y_pct_chg_bp',
    'unemployment_rate_sa_chg_pp'
]

macro_feature_cols = macro_cols[2:]

macro_df = (
    df[macro_cols]
    .drop_duplicates()
    .sort_values('timestep')
    .reset_index(drop=True)
)

macro_df.to_csv(os.path.join(DB_OUTPUT, "macro_data.csv"), index=False)

In [10]:
project_cols = [
    'project_id', 'Project Name',
    'onemap_address',
    'latitude',
    'longitude',
    'Street Name',
    'Postal District',
    'Property Type',
    # 'Completion Date', 
    'Planning Region',
    'Planning Area',
    # 'Tenure Length',
    # 'Tenure Start Date',
    'Large_Dev_200plus',
    'Condo_Age_2026',
    'tenure_remaining_years',
    'tenure_freehold_like',
    'tenure_medium_lease_60_80',
    'tenure_more_than_80',
    'tenure_short_lease_lt60',
    'tenure_unknown',
    # 'project_size_small',
    # 'project_size_medium',
    # 'project_size_large',
]

project_df = (
    df[project_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)

project_df.to_csv(os.path.join(DB_OUTPUT, "project.csv"), index=False)

In [11]:
pre_cols = ['project_id', 'Project Name']
school_cols = [
    col for col in df.columns
    if "School" in col or "Institution" in col
]
pre_school_cols = pre_cols + school_cols
school_df = (
    df[pre_school_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)

school_df.to_csv(os.path.join(DB_OUTPUT, "school.csv"), index=False)

In [12]:
pre_cols = ['project_id', 'Project Name']
mrt_cols = [col for col in df.columns if "_MRT" in col]
pre_mrt_cols = pre_cols + mrt_cols
mrt_df = (
    df[pre_mrt_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)

# convert meters → kilometers
mrt_df[mrt_cols] = mrt_df[mrt_cols] / 1000
mrt_df.to_csv(os.path.join(DB_OUTPUT, "mrt.csv"), index=False)

In [13]:
pre_cols = ['project_id', 'Project Name']
dist_cols = [col for col in df.columns if " dist" in col]
pre_dist_cols = pre_cols + dist_cols
dist_df = (
    df[pre_dist_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)
def convert_to_km(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()
    
    # extract numeric value
    match = re.search(r"[\d\.]+", x)
    if not match:
        return None
    
    value = float(match.group())
    
    # convert based on unit
    if "km" in x:
        return value
    elif "m" in x:
        return value / 1000
    else:
        if value/1000 < 0.01:
            return value
        else:
            return value  # unknown unit
        
for col in dist_cols:
    dist_df[col] = dist_df[col].apply(convert_to_km)
dist_df.to_csv(os.path.join(DB_OUTPUT, "proximity.csv"), index=False)

In [14]:
pre_cols = ['project_id', 'Project Name']
ame_cols = [col for col in df.columns if "Has_" in col]
pre_ame_cols = pre_cols + ame_cols
ame_df = (
    df[pre_ame_cols]
    .drop_duplicates(subset=['project_id'])
    .sort_values('project_id')
    .reset_index(drop=True)
)
# df2 = pd.read_csv(r"dataset\database\amenities.csv")
# ame_df = pd.merge(ame_df, df2, on="Project Name", how="left")
ame_df.to_csv(os.path.join(DB_OUTPUT, "amenities.csv"), index=False)

In [6]:
import numpy as np

data = np.load(r"dataset\mcar_mask_seed123_rate25_v2_new.npz")

# Check what arrays are inside
print(data.files)

['X_train', 'M_obs', 'M_test']


In [7]:
for key in data.files:
    print(key, data[key].shape)

X_train (2259, 313, 1)
M_obs (2259, 313, 1)
M_test (2259, 313, 1)


In [8]:
X_train = data["X_train"]   # or use the actual key name
X_train = np.nan_to_num(X_train, nan=0.0)
X_train.sum()

np.float32(506879.16)

In [9]:
M_obs = data["M_obs"]   # or use the actual key name
M_obs.sum()

np.float64(122948.0)

In [10]:
M_test = data["M_test"]   # or use the actual key name
M_test.sum()

np.float64(40918.0)

In [11]:
X_obs = X_train * M_obs
X_test = X_train * M_test
X_obs.sum() + X_test.sum() - X_train.sum()

np.float64(-0.0053403377532958984)

In [12]:
target = 5.44000006

indices = np.argwhere(np.isclose(X_train, target))

print(indices)

[[   0  286    0]
 [   0  310    0]
 [  31  277    0]
 [  70  262    0]
 [ 123   97    0]
 [ 264  286    0]
 [ 300  192    0]
 [ 300  268    0]
 [ 387  123    0]
 [ 387  147    0]
 [ 387  272    0]
 [ 598  309    0]
 [ 620  222    0]
 [ 649  192    0]
 [ 649  219    0]
 [ 649  268    0]
 [ 649  306    0]
 [ 839  275    0]
 [ 876  143    0]
 [ 876  155    0]
 [ 991  303    0]
 [1099  287    0]
 [1157  205    0]
 [1157  207    0]
 [1157  214    0]
 [1157  215    0]
 [1157  239    0]
 [1189  138    0]
 [1238  109    0]
 [1252   78    0]
 [1265  305    0]
 [1301  159    0]
 [1338  169    0]
 [1354  281    0]
 [1354  297    0]
 [1468  237    0]
 [1484  156    0]
 [1620  136    0]
 [1629  153    0]
 [1731  297    0]
 [1798  273    0]
 [1798  285    0]
 [1833  144    0]
 [1833  167    0]
 [1845  252    0]
 [1862  102    0]
 [1867  280    0]
 [1872   98    0]
 [1880  114    0]
 [1914   95    0]
 [1929  272    0]
 [1929  291    0]
 [1931  277    0]
 [2055  293    0]
 [2157  294    0]]
